### This notebook transforms the aggregated monthly dataset into a modeling-ready dataset by creating temporal, lag, rolling, trend, and interaction features. It also defines the target variable for high-disruption months and prepares the final

# Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load Dataset

In [2]:
# Load the final modeling dataset
final_monthly_df = pd.read_parquet("../data/processed/final_monthly_operational_summary_data.parquet")

# Preview the dataset
final_monthly_df.head()

,YearMonth,Reporting_Airline,Origin,total_completed_flights,dep_delayed_flights,arr_delayed_flights,disrupted_flights,avg_dep_delay,avg_arr_delay,median_dep_delay,...,security_delay_minutes,late_aircraft_delay_minutes,pct_dep_delayed,pct_arr_delayed,pct_disrupted,total_scheduled_flights,cancelled_flights,diverted_flights,pct_cancelled,pct_diverted
0,2023-01,9E,ABE,15,2,2,2,24.666667,11.533333,-5.0,...,0.0,291.0,0.133333,0.133333,0.133333,15,0.0,0.0,0.000000,0.0
1,2023-01,9E,ABY,82,12,13,14,11.073171,8.500000,-5.0,...,0.0,874.0,0.146341,0.158537,0.170732,82,0.0,0.0,0.000000,0.0
2,2023-01,9E,AEX,58,11,10,11,20.758621,10.827586,-5.0,...,0.0,1217.0,0.189655,0.172414,0.189655,60,2.0,0.0,0.033333,0.0
3,2023-01,9E,AGS,26,7,7,7,24.846154,15.038462,-0.5,...,0.0,113.0,0.269231,0.269231,0.269231,26,0.0,0.0,0.000000,0.0
4,2023-01,9E,ALB,107,21,32,34,8.570093,9.130841,-5.0,...,0.0,827.0,0.196262,0.299065,0.317757,110,3.0,0.0,0.027273,0.0


# Create the Target Variable for High-Disruption Months
Rule (Definition):
HighDisruptionMonth = 1 if pct_disrupted is above the 75th percentile 
otherwise 0

## Check the Distribution of Monthly Disruption Rates

In [3]:
# Display summary statistics for the monthly disruption rate
final_monthly_df['pct_disrupted'].describe()

count    55077.000000
mean         0.234520
std          0.127256
min          0.000000
25%          0.150000
50%          0.217391
75%          0.298969
max          1.000000
Name: pct_disrupted, dtype: float64

## Compute the 75th percentile threshold

In [4]:
# Calculate the 75th percentile of the disruption rate
disruption_threshold = final_monthly_df['pct_disrupted'].quantile(0.75)

# Print the threshold value
print(f"High disruption threshold: {disruption_threshold:.4f}")

High disruption threshold: 0.2990


## Create the high-disruption target variable

In [5]:
# Create a binary target for high-disruption months
final_monthly_df['HighDisruptionMonth'] = (
    final_monthly_df['pct_disrupted'] > disruption_threshold
).astype(int)

# Check the class distribution
final_monthly_df['HighDisruptionMonth'].value_counts()

HighDisruptionMonth
0    41308
1    13769
Name: count, dtype: int64

## Preview the new target variable

In [6]:
# Display selected columns including the new target variable
final_monthly_df[
    [
        'YearMonth',
        'Reporting_Airline',
        'Origin',
        'total_scheduled_flights',
        'pct_disrupted',
        'HighDisruptionMonth'
    ]
].head(10)

,YearMonth,Reporting_Airline,Origin,total_scheduled_flights,pct_disrupted,HighDisruptionMonth
0,2023-01,9E,ABE,15,0.133333,0
1,2023-01,9E,ABY,82,0.170732,0
2,2023-01,9E,AEX,60,0.189655,0
3,2023-01,9E,AGS,26,0.269231,0
4,2023-01,9E,ALB,110,0.317757,1
5,2023-01,9E,ATL,2099,0.183792,0
6,2023-01,9E,ATW,32,0.300000,1
7,2023-01,9E,AUS,43,0.209302,0
8,2023-01,9E,AVL,79,0.230769,0
9,2023-01,9E,AZO,62,0.080645,0


## Save the final dataset with target variable

In [7]:
# Save the final dataset including the target variable
final_monthly_df.to_parquet(
    "../data/processed/final_monthly_operational_summary_with_target_data.parquet",
    index=False
)

# Print confirmation message
print("Final monthly dataset with target variable saved successfully.")

Final monthly dataset with target variable saved successfully.


# Create lag features for modeling

## Sort the dataset

In [8]:
# Create a copy of the final monthly dataset for feature engineering
model_df = final_monthly_df.copy()

# Convert YearMonth to datetime format for proper time ordering
model_df['YearMonth'] = pd.to_datetime(model_df['YearMonth'])

# Sort the dataset by airline, origin airport, and month
model_df = model_df.sort_values(
    by=['Reporting_Airline', 'Origin', 'YearMonth']
).reset_index(drop=True)

# Preview the sorted dataset
model_df[['YearMonth', 'Reporting_Airline', 'Origin']].head(10)

,YearMonth,Reporting_Airline,Origin
0,2023-01-01,9E,ABE
1,2023-05-01,9E,ABE
2,2023-06-01,9E,ABE
3,2023-07-01,9E,ABE
4,2023-08-01,9E,ABE
5,2023-09-01,9E,ABE
6,2023-10-01,9E,ABE
7,2023-11-01,9E,ABE
8,2023-12-01,9E,ABE
9,2024-01-01,9E,ABE


## Create lag features

In [9]:
# Create lag features for key operational metrics
model_df['lag_1_pct_disrupted'] = (
    model_df.groupby(['Reporting_Airline', 'Origin'])['pct_disrupted'].shift(1)
)

model_df['lag_2_pct_disrupted'] = (
    model_df.groupby(['Reporting_Airline', 'Origin'])['pct_disrupted'].shift(2)
)

model_df['lag_3_pct_disrupted'] = (
    model_df.groupby(['Reporting_Airline', 'Origin'])['pct_disrupted'].shift(3)
)

model_df['lag_1_pct_cancelled'] = (
    model_df.groupby(['Reporting_Airline', 'Origin'])['pct_cancelled'].shift(1)
)

model_df['lag_2_pct_cancelled'] = (
    model_df.groupby(['Reporting_Airline', 'Origin'])['pct_cancelled'].shift(2)
)

model_df['lag_3_pct_cancelled'] = (
    model_df.groupby(['Reporting_Airline', 'Origin'])['pct_cancelled'].shift(3)
)

model_df['lag_1_pct_diverted'] = (
    model_df.groupby(['Reporting_Airline', 'Origin'])['pct_diverted'].shift(1)
)

model_df['lag_2_pct_diverted'] = (
    model_df.groupby(['Reporting_Airline', 'Origin'])['pct_diverted'].shift(2)
)

model_df['lag_3_pct_diverted'] = (
    model_df.groupby(['Reporting_Airline', 'Origin'])['pct_diverted'].shift(3)
)

model_df['lag_1_avg_dep_delay'] = (
    model_df.groupby(['Reporting_Airline', 'Origin'])['avg_dep_delay'].shift(1)
)

model_df['lag_2_avg_dep_delay'] = (
    model_df.groupby(['Reporting_Airline', 'Origin'])['avg_dep_delay'].shift(2)
)

model_df['lag_3_avg_dep_delay'] = (
    model_df.groupby(['Reporting_Airline', 'Origin'])['avg_dep_delay'].shift(3)
)

model_df['lag_1_avg_arr_delay'] = (
    model_df.groupby(['Reporting_Airline', 'Origin'])['avg_arr_delay'].shift(1)
)

model_df['lag_2_avg_arr_delay'] = (
    model_df.groupby(['Reporting_Airline', 'Origin'])['avg_arr_delay'].shift(2)
)

model_df['lag_3_avg_arr_delay'] = (
    model_df.groupby(['Reporting_Airline', 'Origin'])['avg_arr_delay'].shift(3)
)


# Preview the lag features
model_df[
    [
        'YearMonth',
        'Reporting_Airline',
        'Origin',
        'pct_disrupted',
        'lag_1_pct_disrupted',
        'lag_2_pct_disrupted',
        'lag_3_pct_disrupted',
        'pct_cancelled',
        'lag_1_pct_cancelled',
        'lag_2_pct_cancelled',
        'lag_3_pct_cancelled'
    ]
].head(15)

,YearMonth,Reporting_Airline,Origin,pct_disrupted,lag_1_pct_disrupted,lag_2_pct_disrupted,lag_3_pct_disrupted,pct_cancelled,lag_1_pct_cancelled,lag_2_pct_cancelled,lag_3_pct_cancelled
0,2023-01-01,9E,ABE,0.133333,NaN,NaN,NaN,0.000000,NaN,NaN,NaN
1,2023-05-01,9E,ABE,0.044776,0.133333,NaN,NaN,0.000000,0.000000,NaN,NaN
2,2023-06-01,9E,ABE,0.209302,0.044776,0.133333,NaN,0.000000,0.000000,0.000000,NaN
3,2023-07-01,9E,ABE,0.164706,0.209302,0.044776,0.133333,0.011628,0.000000,0.000000,0.000000
4,2023-08-01,9E,ABE,0.220930,0.164706,0.209302,0.044776,0.022472,0.011628,0.000000,0.000000
5,2023-09-01,9E,ABE,0.100000,0.220930,0.164706,0.209302,0.000000,0.022472,0.011628,0.000000
6,2023-10-01,9E,ABE,0.034884,0.100000,0.220930,0.164706,0.000000,0.000000,0.022472,0.011628
7,2023-11-01,9E,ABE,0.092105,0.034884,0.100000,0.220930,0.000000,0.000000,0.000000,0.022472
8,2023-12-01,9E,ABE,0.097222,0.092105,0.034884,0.100000,0.000000,0.000000,0.000000,0.000000
9,2024-01-01,9E,ABE,0.316456,0.097222,0.092105,0.034884,0.000000,0.000000,0.000000,0.000000


# Create Trend Features

In [10]:
# Create month-to-month trend features
model_df['delta_pct_disrupted'] = (
    model_df['lag_1_pct_disrupted'] - model_df['lag_2_pct_disrupted']
)

model_df['delta_pct_cancelled'] = (
    model_df['lag_1_pct_cancelled'] - model_df['lag_2_pct_cancelled']
)

model_df['delta_pct_diverted'] = (
    model_df['lag_1_pct_diverted'] - model_df['lag_2_pct_diverted']
)

model_df['delta_avg_dep_delay'] = (
    model_df['lag_1_avg_dep_delay'] - model_df['lag_2_avg_dep_delay']
)

model_df['delta_avg_arr_delay'] = (
    model_df['lag_1_avg_arr_delay'] - model_df['lag_2_avg_arr_delay']
)

## Create 2-Step Trend Features

In [11]:
# Create second-order trend (acceleration) for disruption
model_df['delta2_pct_disrupted'] = (
    model_df['lag_1_pct_disrupted']
    - 2 * model_df['lag_2_pct_disrupted']
    + model_df['lag_3_pct_disrupted']
)

# Create second-order trend (acceleration) for cancellation
model_df['delta2_pct_cancelled'] = (
    model_df['lag_1_pct_cancelled']
    - 2 * model_df['lag_2_pct_cancelled']
    + model_df['lag_3_pct_cancelled']
)

# Create second-order trend (acceleration) for arrival delay
model_df['delta2_avg_arr_delay'] = (
    model_df['lag_1_avg_arr_delay']
    - 2 * model_df['lag_2_avg_arr_delay']
    + model_df['lag_3_avg_arr_delay']
)

# Preview the 2-step trend feature
model_df[
    [
        'YearMonth',
        'pct_disrupted',
        'lag_1_pct_disrupted',
        'lag_2_pct_disrupted',
        'lag_3_pct_disrupted',
        'delta_pct_disrupted',
        'delta2_pct_disrupted'
    ]
].head(15)

,YearMonth,pct_disrupted,lag_1_pct_disrupted,lag_2_pct_disrupted,lag_3_pct_disrupted,delta_pct_disrupted,delta2_pct_disrupted
0,2023-01-01,0.133333,NaN,NaN,NaN,NaN,NaN
1,2023-05-01,0.044776,0.133333,NaN,NaN,NaN,NaN
2,2023-06-01,0.209302,0.044776,0.133333,NaN,-0.088557,NaN
3,2023-07-01,0.164706,0.209302,0.044776,0.133333,0.164526,0.253083
4,2023-08-01,0.220930,0.164706,0.209302,0.044776,-0.044596,-0.209123
5,2023-09-01,0.100000,0.220930,0.164706,0.209302,0.056224,0.100821
6,2023-10-01,0.034884,0.100000,0.220930,0.164706,-0.120930,-0.177155
7,2023-11-01,0.092105,0.034884,0.100000,0.220930,-0.065116,0.055814
8,2023-12-01,0.097222,0.092105,0.034884,0.100000,0.057222,0.122338
9,2024-01-01,0.316456,0.097222,0.092105,0.034884,0.005117,-0.052105


# Create Rolling Features

## Create Rolling-3-Month Features

In [12]:
# Create rolling 3-month average features using past values only
model_df['rolling_3_pct_disrupted'] = (
    model_df.groupby(['Reporting_Airline', 'Origin'])['pct_disrupted']
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)

model_df['rolling_3_pct_cancelled'] = (
    model_df.groupby(['Reporting_Airline', 'Origin'])['pct_cancelled']
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)

model_df['rolling_3_avg_dep_delay'] = (
    model_df.groupby(['Reporting_Airline', 'Origin'])['avg_dep_delay']
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)

model_df['rolling_3_avg_arr_delay'] = (
    model_df.groupby(['Reporting_Airline', 'Origin'])['avg_arr_delay']
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)

# Preview the rolling features
model_df[
    [
        'YearMonth',
        'Reporting_Airline',
        'Origin',
        'rolling_3_pct_disrupted',
        'rolling_3_pct_cancelled'
    ]
].head(15)

,YearMonth,Reporting_Airline,Origin,rolling_3_pct_disrupted,rolling_3_pct_cancelled
0,2023-01-01,9E,ABE,NaN,NaN
1,2023-05-01,9E,ABE,0.133333,0.000000
2,2023-06-01,9E,ABE,0.089055,0.000000
3,2023-07-01,9E,ABE,0.129137,0.000000
4,2023-08-01,9E,ABE,0.139595,0.003876
5,2023-09-01,9E,ABE,0.198313,0.011367
6,2023-10-01,9E,ABE,0.161879,0.011367
7,2023-11-01,9E,ABE,0.118605,0.007491
8,2023-12-01,9E,ABE,0.075663,0.000000
9,2024-01-01,9E,ABE,0.074737,0.000000


## Create Rolling Variability Features

In [13]:
# Create rolling three-month standard deviation features using past values only
model_df['rolling_3_std_pct_disrupted'] = (
    model_df.groupby(['Reporting_Airline', 'Origin'])['pct_disrupted']
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=2).std())
)

model_df['rolling_3_std_pct_cancelled'] = (
    model_df.groupby(['Reporting_Airline', 'Origin'])['pct_cancelled']
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=2).std())
)

model_df['rolling_3_std_avg_dep_delay'] = (
    model_df.groupby(['Reporting_Airline', 'Origin'])['avg_arr_delay']
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=2).std())
)

model_df['rolling_3_std_avg_arr_delay'] = (
    model_df.groupby(['Reporting_Airline', 'Origin'])['avg_arr_delay']
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=2).std())
)

# Preview the rolling features
model_df[
    [
        'YearMonth',
        'Reporting_Airline',
        'Origin',
        'rolling_3_std_pct_disrupted',
        'rolling_3_std_pct_cancelled'
    ]
].head(15)

,YearMonth,Reporting_Airline,Origin,rolling_3_std_pct_disrupted,rolling_3_std_pct_cancelled
0,2023-01-01,9E,ABE,NaN,NaN
1,2023-05-01,9E,ABE,NaN,NaN
2,2023-06-01,9E,ABE,0.062619,0.000000
3,2023-07-01,9E,ABE,0.082343,0.000000
4,2023-08-01,9E,ABE,0.085089,0.006713
5,2023-09-01,9E,ABE,0.029679,0.011238
6,2023-10-01,9E,ABE,0.060515,0.011238
7,2023-11-01,9E,ABE,0.094408,0.012974
8,2023-12-01,9E,ABE,0.035536,0.000000
9,2024-01-01,9E,ABE,0.034609,0.000000


# Create Delay-Cause-Per-Flight Features

In [14]:
# Create delay-cause intensity features normalized by scheduled flight volume
model_df['carrier_delay_per_flight'] = (
    model_df['carrier_delay_minutes'] / model_df['total_scheduled_flights']
)

model_df['weather_delay_per_flight'] = (
    model_df['weather_delay_minutes'] / model_df['total_scheduled_flights']
)

model_df['nas_delay_per_flight'] = (
    model_df['nas_delay_minutes'] / model_df['total_scheduled_flights']
)

model_df['late_aircraft_delay_per_flight'] = (
    model_df['late_aircraft_delay_minutes'] / model_df['total_scheduled_flights']
)

# Create Volume Lag Features

In [15]:
# Create lag features for total scheduled flights
model_df['lag_1_total_scheduled_flights'] = (
    model_df.groupby(['Reporting_Airline', 'Origin'])['total_scheduled_flights'].shift(1)
)

model_df['lag_2_total_scheduled_flights'] = (
    model_df.groupby(['Reporting_Airline', 'Origin'])['total_scheduled_flights'].shift(2)
)

model_df['lag_3_total_scheduled_flights'] = (
    model_df.groupby(['Reporting_Airline', 'Origin'])['total_scheduled_flights'].shift(3)
)

# Create month-to-month change in scheduled flight volume
model_df['delta_total_scheduled_flights'] = (
    model_df['lag_1_total_scheduled_flights'] - model_df['lag_2_total_scheduled_flights']
)

# Create a rolling 3-month average of total scheduled flights
model_df['rolling_3_total_scheduled_flights'] = (
    model_df.groupby(['Reporting_Airline', 'Origin'])['total_scheduled_flights']
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)

# Create a rolling 3-month standard deviation of total scheduled flights
model_df['rolling_3_std_total_scheduled_flights'] = (
    model_df.groupby(['Reporting_Airline', 'Origin'])['total_scheduled_flights']
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=2).std())
)

# Preview the volume features
model_df[
    [
        'YearMonth',
        'Reporting_Airline',
        'Origin',
        'total_scheduled_flights',
        'lag_1_total_scheduled_flights',
        'rolling_3_total_scheduled_flights'
    ]
].head(15)

,YearMonth,Reporting_Airline,Origin,total_scheduled_flights,lag_1_total_scheduled_flights,rolling_3_total_scheduled_flights
0,2023-01-01,9E,ABE,15,NaN,NaN
1,2023-05-01,9E,ABE,67,15.0,15.000000
2,2023-06-01,9E,ABE,86,67.0,41.000000
3,2023-07-01,9E,ABE,86,86.0,56.000000
4,2023-08-01,9E,ABE,89,86.0,79.666667
5,2023-09-01,9E,ABE,80,89.0,87.000000
6,2023-10-01,9E,ABE,86,80.0,85.000000
7,2023-11-01,9E,ABE,76,86.0,85.000000
8,2023-12-01,9E,ABE,72,76.0,80.666667
9,2024-01-01,9E,ABE,80,72.0,78.000000


# Create Historical Behaviour Features

## Airline-Level

In [16]:
# Airline-level disruption (previous month)
model_df['airline_prev_pct_disrupted'] = (
    model_df.groupby('Reporting_Airline')['pct_disrupted'].shift(1)
)

# Airline-level cancellation
model_df['airline_prev_pct_cancelled'] = (
    model_df.groupby('Reporting_Airline')['pct_cancelled'].shift(1)
)

## Airport-level 

In [17]:
# Airport-level disruption
model_df['airport_prev_pct_disrupted'] = (
    model_df.groupby('Origin')['pct_disrupted'].shift(1)
)

# Airport-level cancellation
model_df['airport_prev_pct_cancelled'] = (
    model_df.groupby('Origin')['pct_cancelled'].shift(1)
)

# Create Interaction Features

In [18]:
# Disruption × delay interaction
model_df['interaction_disruption_delay'] = (
    model_df['lag_1_pct_disrupted'] * model_df['lag_1_avg_arr_delay']
)

# Disruption × cancellation interaction
model_df['interaction_disruption_cancel'] = (
    model_df['lag_1_pct_disrupted'] * model_df['lag_1_pct_cancelled']
)

# Volume × delay interaction
model_df['interaction_volume_delay'] = (
    model_df['lag_1_total_scheduled_flights'] * model_df['lag_1_avg_arr_delay']
)

# Create Calender Features

In [19]:
# Create calendar-based features from the month
model_df['month_num'] = model_df['YearMonth'].dt.month
model_df['quarter'] = model_df['YearMonth'].dt.quarter

# Preview the calendar features
model_df[['YearMonth', 'month_num', 'quarter']].drop_duplicates().head(12)

,YearMonth,month_num,quarter
0,2023-01-01,1,1
1,2023-05-01,5,2
2,2023-06-01,6,2
3,2023-07-01,7,3
4,2023-08-01,8,3
5,2023-09-01,9,3
6,2023-10-01,10,4
7,2023-11-01,11,4
8,2023-12-01,12,4
9,2024-01-01,1,1


# Check Missing Values in the New Lag Features

In [20]:
# Count missing values in the new lag and rolling features
model_df[
    [
        'lag_1_pct_disrupted',
        'lag_1_pct_cancelled',
        'lag_1_pct_diverted',
        'lag_1_avg_dep_delay',
        'lag_1_avg_arr_delay',
        'lag_2_pct_disrupted',
        'lag_2_pct_cancelled',
        'lag_2_pct_diverted',
        'lag_2_avg_dep_delay',
        'lag_2_avg_arr_delay',
        'rolling_3_pct_disrupted',
        'rolling_3_pct_cancelled',
        'rolling_3_avg_dep_delay',
        'rolling_3_avg_arr_delay',
        'rolling_3_std_pct_disrupted',
        'rolling_3_std_pct_cancelled',
        'rolling_3_std_avg_dep_delay',
        'rolling_3_std_avg_arr_delay',
        'delta_pct_disrupted',
        'delta2_pct_disrupted',
        'delta_pct_cancelled',
        'delta2_pct_cancelled',
        'delta_pct_diverted',
        'delta_avg_dep_delay',
        'delta_avg_arr_delay',
        'delta2_avg_arr_delay',
        'lag_1_total_scheduled_flights',
        'lag_2_total_scheduled_flights',
        'lag_3_total_scheduled_flights',
        'delta_total_scheduled_flights',
        'rolling_3_total_scheduled_flights',
        'rolling_3_std_total_scheduled_flights',
        'airline_prev_pct_disrupted',
        'airline_prev_pct_cancelled',
        'airport_prev_pct_disrupted',
        'airport_prev_pct_cancelled',
        'interaction_disruption_delay',
        'interaction_disruption_cancel',
        'interaction_volume_delay'
    ]
].isnull().sum()

lag_1_pct_disrupted                      1855
lag_1_pct_cancelled                      1855
lag_1_pct_diverted                       1855
lag_1_avg_dep_delay                      1855
lag_1_avg_arr_delay                      1855
lag_2_pct_disrupted                      3689
lag_2_pct_cancelled                      3689
lag_2_pct_diverted                       3689
lag_2_avg_dep_delay                      3689
lag_2_avg_arr_delay                      3689
rolling_3_pct_disrupted                  1855
rolling_3_pct_cancelled                  1855
rolling_3_avg_dep_delay                  1855
rolling_3_avg_arr_delay                  1855
rolling_3_std_pct_disrupted              3689
rolling_3_std_pct_cancelled              3689
rolling_3_std_avg_dep_delay              3689
rolling_3_std_avg_arr_delay              3689
delta_pct_disrupted                      3689
delta2_pct_disrupted                     5502
delta_pct_cancelled                      3689
delta2_pct_cancelled              

# Handling Missing Values

## Fill Missing Values

In [21]:
# Fill missing airline features
model_df['airline_prev_pct_disrupted'].fillna(0, inplace=True)
model_df['airline_prev_pct_cancelled'].fillna(0, inplace=True)

# Fill missing airport features
model_df['airport_prev_pct_disrupted'].fillna(0, inplace=True)
model_df['airport_prev_pct_cancelled'].fillna(0, inplace=True)

C:\Users\Administrator\AppData\Local\Temp\ipykernel_19708\2152616496.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  model_df['airline_prev_pct_disrupted'].fillna(0, inplace=True)
C:\Users\Administrator\AppData\Local\Temp\ipykernel_19708\2152616496.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values alwa

In [22]:
model_df[
    [
        'airline_prev_pct_disrupted',
        'airport_prev_pct_disrupted'
    ]
].isnull().sum()

airline_prev_pct_disrupted    0
airport_prev_pct_disrupted    0
dtype: int64

## Drop Rows Without Prior History

In [23]:
# Drop rows without sufficient history for all key features
model_df = model_df.dropna(
    subset=[
        'lag_1_pct_disrupted',
        'lag_2_pct_disrupted',
        'lag_3_pct_disrupted'
    ]
).copy()

# Display new shape
model_df.shape

(49575, 75)

In [24]:
# Confirm key features are now clean
model_df[
    [
        'lag_1_pct_disrupted',
        'lag_2_pct_disrupted',
        'lag_3_pct_disrupted',
        'delta_pct_disrupted',
        'delta2_pct_disrupted'
    ]
].isnull().sum()

lag_1_pct_disrupted     0
lag_2_pct_disrupted     0
lag_3_pct_disrupted     0
delta_pct_disrupted     0
delta2_pct_disrupted    0
dtype: int64

# Check the target distribution 

In [25]:
# Check the class distribution after removing rows without prior history
model_df['HighDisruptionMonth'].value_counts()

HighDisruptionMonth
0    37326
1    12249
Name: count, dtype: int64

# Save Modeling Dataset

In [26]:
# Save the final modeling dataset with lag features
model_df.to_parquet("../data/processed/modeling_dataset.parquet", index=False)

# Print confirmation message
print("Modeling dataset with lag features saved successfully.")

Modeling dataset with lag features saved successfully.
